# Fase 3 — memória conversacional

Este notebook usa o mesmo serviço Python do agente. O teste local prova persistência e isolamento; **não demonstra compreensão por LLM**. A conversa real com o agente requer API e deve ser registrada separadamente.


In [ ]:
from pathlib import Path
import sys, tempfile
raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.memoria import MemoriaConversas


## 1. Uma conversa, três turnos e janela curta
Cada turno completo é salvo somente depois que a resposta existe. A janela de dois turnos limita o contexto enviado ao modelo.


In [ ]:
pasta = tempfile.TemporaryDirectory()
caminho = Path(pasta.name) / 'memoria.sqlite3'
ana, bruno = Contexto('demo', 'ana'), Contexto('demo', 'bruno')
with Banco(caminho) as banco:
    banco.preparar_demo()
    memoria = MemoriaConversas(banco)
    sessao = memoria.iniciar(ana)
    memoria.registrar(ana, sessao, 'Tenho reunião com Alfa', 'Entendi; qual data?')
    memoria.registrar(ana, sessao, 'Amanhã às 14h', 'Preciso da data absoluta para agendar.')
    memoria.registrar(ana, sessao, 'Qual cliente mencionei?', 'Você mencionou Alfa.')
    janela = memoria.mensagens(ana, sessao, limite=2)
    assert len(janela) == 4 and janela[0]['content'] == 'Amanhã às 14h'
    print(janela)


## 2. Reabrir, separar usuários e começar do zero
Reabrir o banco recupera os turnos. `/nova` no terminal encerra a sessão atual e cria outra vazia; o histórico antigo permanece auditável. O usuário é de demonstração, sem autenticação real.


In [ ]:
with Banco(caminho) as banco:
    memoria = MemoriaConversas(banco)
    assert memoria.ultima_ativa(ana) == sessao
    assert len(memoria.ultimos(ana, sessao)) == 3
    assert memoria.ultima_ativa(bruno) is None
    try:
        memoria.ultimos(bruno, sessao)
    except PermissionError:
        print('Acesso de outro usuário negado, como esperado.')
    else:
        raise AssertionError('Falha de isolamento')
    memoria.encerrar(ana, sessao)
    nova = memoria.iniciar(ana)
    assert memoria.ultimos(ana, nova) == []
    print('Nova conversa vazia:', nova)
pasta.cleanup()


## 3. Teste real pendente
No menu 9 de `executar.py`, use a API para mencionar um cliente, pergunte depois `Qual cliente mencionei?`, saia com `/voltar`, reabra o programa com o mesmo `--usuario` e repita a pergunta. Use `/nova` e pergunte novamente: o agente não deve recuperar a conversa anterior como contexto. Registre respostas e chamadas observadas; este notebook não substitui essa evidência.
